In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/outbreak_dataset_clean_sorted (1).csv")

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2093 entries, 0 to 2092
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   state             2093 non-null   object
 1   district          2093 non-null   object
 2   disease           2093 non-null   object
 3   cases             2093 non-null   int64 
 4   deaths            2093 non-null   int64 
 5   start_date        2093 non-null   object
 6   report_date       2093 non-null   object
 7   status            2093 non-null   object
 8   comments          876 non-null    object
 9   year              2093 non-null   int64 
 10  month             2093 non-null   int64 
 11  week              2093 non-null   int64 
 12  district_clean    2093 non-null   object
 13  district_matched  2093 non-null   object
dtypes: int64(5), object(9)
memory usage: 229.1+ KB


In [3]:
df.columns

Index(['state', 'district', 'disease', 'cases', 'deaths', 'start_date',
       'report_date', 'status', 'comments', 'year', 'month', 'week',
       'district_clean', 'district_matched'],
      dtype='object')

In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

In [5]:
df.replace({
    r'\*': np.nan,
    r'\(': '',
    r'\)': '',
    r',': ''
}, regex=True, inplace=True)

In [6]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

In [7]:
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['report_date'] = pd.to_datetime(df['report_date'], errors='coerce')

In [8]:
df['comments'].fillna('no_comment', inplace=True)

/tmp/ipykernel_283/2297901852.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['comments'].fillna('no_comment', inplace=True)


In [9]:
df.drop(columns=['comments'], inplace=True)

In [10]:
# 1. purana district hata do
df.drop(columns=['district'], inplace=True)

# 2. district_matched ko rename karo
df.rename(columns={'district_matched': 'district'}, inplace=True)

# 3. district_clean bhi hata do
df.drop(columns=['district_clean'], inplace=True)

In [11]:
df = df.loc[:, ~df.columns.duplicated()]

In [12]:
df

,state,disease,cases,deaths,start_date,report_date,status,year,month,week,district
0,kerala,hepatitis,22,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,ernakulam
1,kerala,food poisoning,182,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,kozhikkode
2,kerala,chickenpox,30,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,kozhikkode
3,telangana,acute diarrheal disease,104,0,2026-01-30,2026-01-30,under surveillance,2026,1,5,suryapet
4,kerala,hepatitis,22,0,2026-01-30,2026-01-31,under surveillance,2026,1,5,idukki
...,...,...,...,...,...,...,...,...,...,...,...
2088,odisha,acute diarrheal disease,0,0,2024-12-30,2024-12-30,cases were reported from village basudebpur su...,2024,12,1,deoghar
2089,maharashtra,acute diarrheal disease,0,0,2024-12-30,2024-12-30,cases were reported from village dighode sub-d...,2024,12,1,raigarh
2090,maharashtra,acute diarrheal disease,0,0,2024-12-30,2024-12-30,cases were reported from village adgaon sub-di...,2024,12,1,amravati
2091,kerala,food poisoning,0,0,2024-12-30,2024-12-30,cases were reported from darool uloom madrasa ...,2024,12,1,kollam


In [13]:
df.columns

Index(['state', 'disease', 'cases', 'deaths', 'start_date', 'report_date',
       'status', 'year', 'month', 'week', 'district'],
      dtype='object')

In [15]:
df["district"].unique()

array(['ernakulam', 'kozhikkode', 'suryapet', 'idukki', 'dhar',
       'palakkad', 'begusarai', 'baramula', 'purba bardhaman', 'east',
       'umaria', 'gwalior', 'purnia', 'gadchiroli', 'jharsuguda',
       'perambalur', 'chamoli', 'ganjam', 'bhavnagar', 'kottayam',
       'dewas', 'thiruvananthapuram', 'kamrup', 'coimbatore', 'prakasam',
       'thrissur', 'nabarangapur', 'farrukhabad', 'hapur', 'rajkot',
       'panch mahals', 'viluppuram', 'ashoknagar', 'rajnandgaon',
       'raigarh', 'ahmedabad', 'lucknow', 'hassan', 'west',
       'janjgir champa', 'kannur', 'bidar', 'nagapattinam', 'katihar',
       'morena', 'saran', 'namakkal', 'jamtara', 'gadag', 'latehar',
       'thane', 'sundargarh', 'mandi', 'ghaziabad', 'biswanath', 'salem',
       'malappuram', 'north', 'narsinghpur', 'kalahandi', 'burhanpur',
       'south', 'kendujhar', 'nanded', 'jalpaiguri', 'ludhiana',
       'papum pare', 'pudukkottai', 'chandrapur', 'kokrajhar',
       'gautam buddha nagar', 'jamnagar', 'kendrap

In [16]:
df1 = pd.read_csv("/content/nfhs_ML_Pipeline_dataset.csv")

In [18]:
# outbreak vs nfhs
missing_in_nfhs = set(df['district']) - set(df1['district'])

# nfhs vs outbreak
missing_in_outbreak = set(df1['district']) - set(df['district'])

print("Not in NFHS:", len(missing_in_nfhs))
print("Not in Outbreak:", len(missing_in_outbreak))

Not in NFHS: 0
Not in Outbreak: 251


In [19]:
df

,state,disease,cases,deaths,start_date,report_date,status,year,month,week,district
0,kerala,hepatitis,22,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,ernakulam
1,kerala,food poisoning,182,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,kozhikkode
2,kerala,chickenpox,30,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,kozhikkode
3,telangana,acute diarrheal disease,104,0,2026-01-30,2026-01-30,under surveillance,2026,1,5,suryapet
4,kerala,hepatitis,22,0,2026-01-30,2026-01-31,under surveillance,2026,1,5,idukki
...,...,...,...,...,...,...,...,...,...,...,...
2088,odisha,acute diarrheal disease,0,0,2024-12-30,2024-12-30,cases were reported from village basudebpur su...,2024,12,1,deoghar
2089,maharashtra,acute diarrheal disease,0,0,2024-12-30,2024-12-30,cases were reported from village dighode sub-d...,2024,12,1,raigarh
2090,maharashtra,acute diarrheal disease,0,0,2024-12-30,2024-12-30,cases were reported from village adgaon sub-di...,2024,12,1,amravati
2091,kerala,food poisoning,0,0,2024-12-30,2024-12-30,cases were reported from darool uloom madrasa ...,2024,12,1,kollam


In [20]:
df['disease'] = df['disease'].str.strip().str.lower()
df['status'] = df['status'].str.strip().str.lower()

In [21]:
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['report_date'] = pd.to_datetime(df['report_date'], errors='coerce')

In [22]:
df = df[~((df['cases'] == 0) & (df['deaths'] == 0))]

In [23]:
df.drop_duplicates(inplace=True)

/tmp/ipykernel_283/3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [24]:
df.head()

,state,disease,cases,deaths,start_date,report_date,status,year,month,week,district
0,kerala,hepatitis,22,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,ernakulam
1,kerala,food poisoning,182,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,kozhikkode
2,kerala,chickenpox,30,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,kozhikkode
3,telangana,acute diarrheal disease,104,0,2026-01-30,2026-01-30,under surveillance,2026,1,5,suryapet
4,kerala,hepatitis,22,0,2026-01-30,2026-01-31,under surveillance,2026,1,5,idukki


In [25]:
df.to_csv("outbreak1.csv", index=False)

# Feature eng

In [56]:
import pandas as pd

df_outbreak = pd.read_csv("outbreak1.csv")
df_outbreak.head()

,state,disease,cases,deaths,start_date,report_date,status,year,month,week,district
0,kerala,hepatitis,22,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,ernakulam
1,kerala,food poisoning,182,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,kozhikkode
2,kerala,chickenpox,30,0,2026-01-31,2026-01-31,under surveillance,2026,1,5,kozhikkode
3,telangana,acute diarrheal disease,104,0,2026-01-30,2026-01-30,under surveillance,2026,1,5,suryapet
4,kerala,hepatitis,22,0,2026-01-30,2026-01-31,under surveillance,2026,1,5,idukki


In [57]:
df_outbreak.columns


Index(['state', 'disease', 'cases', 'deaths', 'start_date', 'report_date',
       'status', 'year', 'month', 'week', 'district'],
      dtype='object')

In [58]:
df_outbreak.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1003 entries, 0 to 1002
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   state        1003 non-null   object
 1   disease      1003 non-null   object
 2   cases        1003 non-null   int64 
 3   deaths       1003 non-null   int64 
 4   start_date   1003 non-null   object
 5   report_date  1003 non-null   object
 6   status       1003 non-null   object
 7   year         1003 non-null   int64 
 8   month        1003 non-null   int64 
 9   week         1003 non-null   int64 
 10  district     1003 non-null   object
dtypes: int64(5), object(6)
memory usage: 86.3+ KB


In [59]:
df_outbreak.columns = df_outbreak.columns.str.lower().str.replace(" ", "_")

In [60]:
df.columns

Index(['state', 'disease', 'cases', 'deaths', 'start_date', 'report_date',
       'status', 'year', 'month', 'week', 'district', 'day_of_week',
       'is_weekend', 'cases_lag_1', 'cases_lag_2', 'cases_lag_3',
       'cases_rolling_mean_3', 'cases_rolling_mean_5', 'case_change',
       'case_growth_rate', 'spike_flag', 'death_rate', 'severity_score',
       'top_disease', 'top_disease_total_cases', 'top_disease_total_deaths',
       'avg_severity_score', 'total_spikes', 'disease_frequency',
       'recent_trend', 'disease_risk_score'],
      dtype='object')

In [61]:
df_outbreak['start_date'] = pd.to_datetime(df_outbreak['start_date'], errors='coerce')
df_outbreak['report_date'] = pd.to_datetime(df_outbreak['report_date'], errors='coerce')

In [62]:
df_outbreak['year'] = df_outbreak['report_date'].dt.year
df_outbreak['month'] = df_outbreak['report_date'].dt.month
df_outbreak['week'] = df_outbreak['report_date'].dt.isocalendar().week

In [63]:
# time features
df['year'] = df['report_date'].dt.year
df['month'] = df['report_date'].dt.month
df['week'] = df['report_date'].dt.isocalendar().week

# advanced
df['day_of_week'] = df['report_date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

In [64]:
df['cases'] = pd.to_numeric(df['cases'], errors='coerce')
df['deaths'] = pd.to_numeric(df['deaths'], errors='coerce')

df = df.dropna(subset=['cases'])
df['deaths'] = df['deaths'].fillna(0)

In [65]:
df = df.sort_values(by=['district','disease','report_date'])

df['cases_lag_1'] = df.groupby(['district','disease'])['cases'].shift(1)
df['cases_lag_2'] = df.groupby(['district','disease'])['cases'].shift(2)
df['cases_lag_3'] = df.groupby(['district','disease'])['cases'].shift(3)

In [66]:
df['cases_rolling_mean_3'] = df.groupby(['district','disease'])['cases'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

df['cases_rolling_mean_5'] = df.groupby(['district','disease'])['cases'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean()
)

In [67]:
df['case_change'] = df['cases'] - df['cases_lag_1']

df['case_growth_rate'] = df['case_change'] / (df['cases_lag_1'] + 1)

df['spike_flag'] = (df['case_growth_rate'] > 0.5).astype(int)

In [68]:
df['death_rate'] = df['deaths'] / (df['cases'] + 1)

df['severity_score'] = (
    0.7 * df['cases'] +
    0.3 * df['deaths']
)

In [69]:
summary_df = df.groupby(['state','district','disease']).agg({
    'cases': 'sum',
    'deaths': 'sum',
    'severity_score': 'mean',
    'spike_flag': 'sum'
}).reset_index()

In [70]:
top_disease = summary_df.loc[
    summary_df.groupby('district')['cases'].idxmax()
]

top_disease.rename(columns={
    'disease': 'top_disease',
    'cases': 'top_disease_cases'
}, inplace=True)

In [71]:
summary_df = df.groupby(['state','district','disease']).agg({
    'cases': 'sum',
    'deaths': 'sum',
    'severity_score': 'mean',
    'spike_flag': 'sum'
}).reset_index()

In [72]:
top_disease_df = summary_df.loc[
    summary_df.groupby('district')['cases'].idxmax()
]

In [73]:
top_disease_df = top_disease_df.rename(columns={
    'disease': 'top_disease',
    'cases': 'top_disease_total_cases',
    'deaths': 'top_disease_total_deaths',
    'severity_score': 'avg_severity_score',
    'spike_flag': 'total_spikes'
})

In [74]:
df = pd.merge(
    df,
    top_disease_df[['district','top_disease','top_disease_total_cases',
                    'top_disease_total_deaths','avg_severity_score','total_spikes']],
    on='district',
    how='left'
)

In [75]:
df_final = df.sort_values('report_date').groupby(['district','disease']).tail(1)

In [76]:
df_final

,state,disease,cases,deaths,start_date,report_date,status,year,month,week,...,avg_severity_score_x,total_spikes_x,disease_frequency,recent_trend,disease_risk_score,top_disease_y,top_disease_total_cases_y,top_disease_total_deaths_y,avg_severity_score_y,total_spikes_y
434,madhya pradesh,acute diarrheal disease,0.0080,0.0,2025-01-16,2025-01-16,cases of loose motion and vomiting were report...,2025,1,3,...,7.700000,0,1,0.000000,0.017986,acute diarrheal disease,0.0080,0.00,0.005600,0
69,west bengal,hepatitis,0.0392,0.0,2025-01-19,2025-01-19,cases were reported from villages kuldoba and ...,2025,1,3,...,35.000000,0,1,0.000000,0.069302,hepatitis,0.0392,0.00,0.027440,0
54,karnataka,measles,0.0208,0.0,2025-06-29,2025-06-29,cases were reported from lamani tanda and sidd...,2025,6,26,...,18.900000,0,1,0.000000,0.039039,measles,0.0208,0.00,0.014560,0
593,punjab,hepatitis,0.0104,0.0,2025-07-02,2025-07-02,cases of loss of appetite nausea and yellow co...,2025,7,27,...,86.800000,0,1,0.000000,0.021934,food poisoning,0.0984,0.00,0.068880,0
653,arunachal pradesh,typhoid,0.0616,0.0,2025-07-08,2025-07-08,cases were reported from township chownkham su...,2025,7,28,...,68.600000,0,1,0.000000,0.106144,dengue,0.0776,0.00,0.054320,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
687,kerala,chickenpox,0.0272,0.0,2026-01-30,2026-01-31,under surveillance,2026,1,5,...,56.875000,0,7,0.010133,0.258856,acute gastroenteritis,0.2568,0.00,0.044940,0
283,kerala,hepatitis,0.0168,0.0,2026-01-31,2026-01-31,under surveillance,2026,1,5,...,41.164286,5,6,0.006667,0.238189,acute gastroenteritis,0.6464,0.12,0.034891,0
550,kerala,chickenpox,0.0232,0.0,2026-01-31,2026-01-31,under surveillance,2026,1,5,...,82.950000,1,1,0.000000,0.042986,food poisoning,0.1880,0.00,0.065800,0
377,kerala,hepatitis,0.0168,0.0,2026-01-30,2026-01-31,under surveillance,2026,1,5,...,38.966667,1,1,0.000000,0.032460,food poisoning,0.1312,0.00,0.030613,0


In [77]:
df_final['cases_lag_1'] = df_final['cases_lag_1'].fillna(0)

df_final['case_change'] = df_final['case_change'].fillna(0)

df_final['case_growth_rate'] = df_final['case_growth_rate'].fillna(0)

In [78]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

cols_to_scale = [
    'cases', 'deaths',
    'cases_lag_1', 'cases_lag_2', 'cases_lag_3',
    'cases_rolling_mean_3', 'cases_rolling_mean_5',
    'case_change', 'case_growth_rate',
    'death_rate', 'severity_score'
]

df_final[cols_to_scale] = scaler.fit_transform(df_final[cols_to_scale])

In [79]:
df_final['disease_frequency'] = df_final.groupby(['district','disease'])['cases'].transform('count')

In [80]:
df_final['recent_trend'] = df_final['cases'] - df_final['cases_rolling_mean_3']

In [81]:
df_final['disease_risk_score'] = (
    0.4 * df_final['cases'] +
    0.2 * df_final['case_growth_rate'] +
    0.2 * df_final['severity_score'] +
    0.2 * df_final['spike_flag']
)

In [82]:
df_final

,state,disease,cases,deaths,start_date,report_date,status,year,month,week,...,avg_severity_score_x,total_spikes_x,disease_frequency,recent_trend,disease_risk_score,top_disease_y,top_disease_total_cases_y,top_disease_total_deaths_y,avg_severity_score_y,total_spikes_y
434,madhya pradesh,acute diarrheal disease,0.021930,0.0,2025-01-16,2025-01-16,cases of loose motion and vomiting were report...,2025,1,3,...,7.700000,0,1,-0.000988,0.078296,acute diarrheal disease,0.0080,0.00,0.005600,0
69,west bengal,hepatitis,0.107456,0.0,2025-01-19,2025-01-19,cases were reported from villages kuldoba and ...,2025,1,3,...,35.000000,0,1,-0.004843,0.129612,hepatitis,0.0392,0.00,0.027440,0
54,karnataka,measles,0.057018,0.0,2025-06-29,2025-06-29,cases were reported from lamani tanda and sidd...,2025,6,26,...,18.900000,0,1,-0.002570,0.099349,measles,0.0208,0.00,0.014560,0
593,punjab,hepatitis,0.028509,0.0,2025-07-02,2025-07-02,cases of loss of appetite nausea and yellow co...,2025,7,27,...,86.800000,0,1,-0.001285,0.082244,food poisoning,0.0984,0.00,0.068880,0
653,arunachal pradesh,typhoid,0.168860,0.0,2025-07-08,2025-07-08,cases were reported from township chownkham su...,2025,7,28,...,68.600000,0,1,-0.007611,0.166454,dengue,0.0776,0.00,0.054320,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
687,kerala,chickenpox,0.074561,0.0,2026-01-30,2026-01-31,under surveillance,2026,1,5,...,56.875000,0,1,0.025669,0.116825,acute gastroenteritis,0.2568,0.00,0.044940,0
283,kerala,hepatitis,0.046053,0.0,2026-01-31,2026-01-31,under surveillance,2026,1,5,...,41.164286,5,1,0.017023,0.096402,acute gastroenteritis,0.6464,0.12,0.034891,0
550,kerala,chickenpox,0.063596,0.0,2026-01-31,2026-01-31,under surveillance,2026,1,5,...,82.950000,1,1,-0.002866,0.103296,food poisoning,0.1880,0.00,0.065800,0
377,kerala,hepatitis,0.046053,0.0,2026-01-30,2026-01-31,under surveillance,2026,1,5,...,38.966667,1,1,-0.002076,0.092770,food poisoning,0.1312,0.00,0.030613,0


In [88]:
# remove all _x columns
df_final = df_final[[col for col in df_final.columns if not col.endswith('_x')]]

In [89]:
df_final.columns = df_final.columns.str.replace('_y', '', regex=False)

In [90]:
print(df_final.columns)

Index(['state', 'disease', 'cases', 'deaths', 'start_date', 'report_date',
       'status', 'year', 'month', 'week', 'district', 'day_of_week',
       'is_weekend', 'cases_lag_1', 'cases_lag_2', 'cases_lag_3',
       'cases_rolling_mean_3', 'cases_rolling_mean_5', 'case_change',
       'case_growth_rate', 'spike_flag', 'death_rate', 'severity_score',
       'disease_frequency', 'recent_trend', 'disease_risk_score',
       'top_disease', 'top_disease_total_cases', 'top_disease_total_deaths',
       'avg_severity_score', 'total_spikes'],
      dtype='object')


In [91]:
df_final['is_top_disease'] = (df_final['disease'] == df_final['top_disease']).astype(int)

/tmp/ipykernel_283/3304797866.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['is_top_disease'] = (df_final['disease'] == df_final['top_disease']).astype(int)


In [92]:
df_final.columns

Index(['state', 'disease', 'cases', 'deaths', 'start_date', 'report_date',
       'status', 'year', 'month', 'week', 'district', 'day_of_week',
       'is_weekend', 'cases_lag_1', 'cases_lag_2', 'cases_lag_3',
       'cases_rolling_mean_3', 'cases_rolling_mean_5', 'case_change',
       'case_growth_rate', 'spike_flag', 'death_rate', 'severity_score',
       'disease_frequency', 'recent_trend', 'disease_risk_score',
       'top_disease', 'top_disease_total_cases', 'top_disease_total_deaths',
       'avg_severity_score', 'total_spikes', 'is_top_disease'],
      dtype='object')

In [95]:
df_final['disease_frequency'] = df_final['disease_frequency'] / df_final['disease_frequency'].max()

/tmp/ipykernel_283/598667592.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['disease_frequency'] = df_final['disease_frequency'] / df_final['disease_frequency'].max()


In [96]:
df_final['enhanced_disease_risk_score'] = (
    0.35 * df_final['cases'] +
    0.2 * df_final['case_growth_rate'] +
    0.15 * df_final['severity_score'] +
    0.15 * df_final['spike_flag'] +
    0.15 * df_final['disease_frequency']
)

/tmp/ipykernel_283/119234122.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['enhanced_disease_risk_score'] = (


In [97]:
def trend_label(x):
    if x > 0.05:
        return "Increasing"
    elif x < -0.05:
        return "Decreasing"
    else:
        return "Stable"

df_final['disease_trend'] = df_final['recent_trend'].apply(trend_label)

/tmp/ipykernel_283/764064492.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['disease_trend'] = df_final['recent_trend'].apply(trend_label)


In [98]:
df_final

,state,disease,cases,deaths,start_date,report_date,status,year,month,week,...,recent_trend,disease_risk_score,top_disease,top_disease_total_cases,top_disease_total_deaths,avg_severity_score,total_spikes,is_top_disease,enhanced_disease_risk_score,disease_trend
434,madhya pradesh,acute diarrheal disease,0.021930,0.0,2025-01-16,2025-01-16,cases of loose motion and vomiting were report...,2025,1,3,...,-0.000988,0.078296,acute diarrheal disease,0.0080,0.00,0.005600,0,1,0.086818,Stable
69,west bengal,hepatitis,0.107456,0.0,2025-01-19,2025-01-19,cases were reported from villages kuldoba and ...,2025,1,3,...,-0.004843,0.129612,hepatitis,0.0392,0.00,0.027440,0,1,0.129581,Stable
54,karnataka,measles,0.057018,0.0,2025-06-29,2025-06-29,cases were reported from lamani tanda and sidd...,2025,6,26,...,-0.002570,0.099349,measles,0.0208,0.00,0.014560,0,1,0.104362,Stable
593,punjab,hepatitis,0.028509,0.0,2025-07-02,2025-07-02,cases of loss of appetite nausea and yellow co...,2025,7,27,...,-0.001285,0.082244,food poisoning,0.0984,0.00,0.068880,0,0,0.090107,Stable
653,arunachal pradesh,typhoid,0.168860,0.0,2025-07-08,2025-07-08,cases were reported from township chownkham su...,2025,7,28,...,-0.007611,0.166454,dengue,0.0776,0.00,0.054320,0,0,0.160283,Stable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
687,kerala,chickenpox,0.074561,0.0,2026-01-30,2026-01-31,under surveillance,2026,1,5,...,0.025669,0.116825,acute gastroenteritis,0.2568,0.00,0.044940,0,0,0.184368,Stable
283,kerala,hepatitis,0.046053,0.0,2026-01-31,2026-01-31,under surveillance,2026,1,5,...,0.017023,0.096402,acute gastroenteritis,0.6464,0.12,0.034891,0,0,0.156082,Stable
550,kerala,chickenpox,0.063596,0.0,2026-01-31,2026-01-31,under surveillance,2026,1,5,...,-0.002866,0.103296,food poisoning,0.1880,0.00,0.065800,0,0,0.107651,Stable
377,kerala,hepatitis,0.046053,0.0,2026-01-30,2026-01-31,under surveillance,2026,1,5,...,-0.002076,0.092770,food poisoning,0.1312,0.00,0.030613,0,0,0.098879,Stable


In [99]:
df_final.to_csv("outbreak_ML_pipeline_dataset.csv", index=False)